# 🌙 Somnia v2 — Notebook 04 : CNN 1D (Détection apnée)

**Objectif :** Améliorer la généralisation inter-sujets du modèle ECG

**Motivation :** Le Random Forest sur features ECG donne AUC=0.70 en validation par sujet.  
Un CNN 1D travaillant directement sur le **signal brut** apprend des patterns temporels  
plus robustes que des features handcrafted, ce qui améliore la généralisation.

**Plan :**
1. Préparation des données brutes (signal ECG 6000 points)
2. Architecture CNN 1D
3. Entraînement + MLflow tracking
4. Évaluation — split aléatoire + split par sujet
5. Comparaison RF vs CNN
6. Sauvegarde modèle production

**Auteur :** Marine Del Dicque  
**Contexte :** AIA Jedha — Bloc 4

## 0. Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import joblib
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ML classique
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              roc_curve, classification_report,
                              confusion_matrix)
from sklearn.preprocessing import StandardScaler

# ECG
import wfdb
from scipy.signal import find_peaks, butter, filtfilt

# MLflow
import mlflow
import mlflow.pytorch

# Chemins
DATA_RAW_ECG = Path('../data/raw_apnea')
MODELS_DIR   = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)
Path('../data/figures').mkdir(exist_ok=True)

# MLflow
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('Somnia_v2')

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Constantes
FS          = 100
SEG_LEN     = 60       # secondes
SEG_SAMPLES = FS * SEG_LEN  # 6000 points

PALETTE = ['#048A81', '#E84855']

print('✅ Imports OK')
print(f'🖥️  Device : {DEVICE}')
print(f'🔥 PyTorch : {torch.__version__}')
print(f'📊 MLflow experiment : Somnia_v2')

## 1. Chargement des données brutes ECG

Contrairement au RF qui travaillait sur 16 features extraites,  
le CNN reçoit directement le **signal ECG brut de 6000 points**.

In [ ]:
def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100, order=4):
    """
    Filtre passe-bande pour nettoyer le signal ECG.
    Supprime la dérive baseline (<0.5 Hz) et le bruit haute fréquence (>40 Hz).
    """
    nyq  = fs / 2
    low  = lowcut  / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


def load_raw_ecg_subject(subj_id, data_dir, fs=100, seg_len=60):
    """
    Charge les segments ECG bruts d'un sujet.
    
    Returns
    -------
    segments : np.array (n_segments, seg_samples)
    labels   : np.array (n_segments,) — 0=Normal, 1=Apnée
    """
    rec = wfdb.rdrecord(str(data_dir / subj_id))
    ann = wfdb.rdann(str(data_dir / subj_id), 'apn')
    ecg = rec.p_signal[:, 0]
    
    # Filtre passe-bande
    ecg = bandpass_filter(ecg, fs=fs)
    
    # Normalisation z-score intra-sujet
    ecg = (ecg - np.mean(ecg)) / (np.std(ecg) + 1e-8)
    
    seg_samples = fs * seg_len
    segments, labels = [], []
    label_map = {'A': 1, 'N': 0}
    
    for i, sym in enumerate(ann.symbol):
        if sym not in label_map:
            continue
        start = i * seg_samples
        end   = start + seg_samples
        if end <= len(ecg):
            segments.append(ecg[start:end].astype(np.float32))
            labels.append(label_map[sym])
    
    return np.array(segments), np.array(labels, dtype=np.int64)


# Split par sujet (même que notebook 03)
TRAIN_SUBJECTS = ['a01','a02','a03','a04','a05','a06','a07',
                  'a08','a09','a10','a11','a13',
                  'b01','b02','b03','c01','c02','c03','c04']
TEST_SUBJECTS  = ['a12','a14','a15','a16','a17','a18',
                  'b04','b05','c05','c06','c08','c10']

# Chargement
print('Chargement des signaux ECG bruts...')
X_train_raw, y_train_raw = [], []
X_test_raw,  y_test_raw  = [], []

for subj in TRAIN_SUBJECTS:
    try:
        X_s, y_s = load_raw_ecg_subject(subj, DATA_RAW_ECG)
        X_train_raw.append(X_s)
        y_train_raw.append(y_s)
        print(f'  Train {subj} : {len(y_s)} seg | '
              f'Apnée={sum(y_s==1)} ({sum(y_s==1)/len(y_s)*100:.0f}%)')
    except Exception as e:
        print(f'  Train {subj} ❌ {e}')

for subj in TEST_SUBJECTS:
    try:
        X_s, y_s = load_raw_ecg_subject(subj, DATA_RAW_ECG)
        X_test_raw.append(X_s)
        y_test_raw.append(y_s)
        print(f'  Test  {subj} : {len(y_s)} seg | '
              f'Apnée={sum(y_s==1)} ({sum(y_s==1)/len(y_s)*100:.0f}%)')
    except Exception as e:
        print(f'  Test  {subj} ❌ {e}')

X_train_raw = np.vstack(X_train_raw)
y_train_raw = np.concatenate(y_train_raw)
X_test_raw  = np.vstack(X_test_raw)
y_test_raw  = np.concatenate(y_test_raw)

print(f'\n✅ Train : {X_train_raw.shape} | '
      f'Apnée={sum(y_train_raw==1)} ({sum(y_train_raw==1)/len(y_train_raw)*100:.1f}%)')
print(f'   Test  : {X_test_raw.shape}  | '
      f'Apnée={sum(y_test_raw==1)} ({sum(y_test_raw==1)/len(y_test_raw)*100:.1f}%)')

## 2. Dataset PyTorch

In [ ]:
class ECGDataset(Dataset):
    """
    Dataset PyTorch pour les signaux ECG.
    
    Input  : signal ECG brut (6000 points)
    Output : label binaire (0=Normal, 1=Apnée)
    """
    def __init__(self, X, y):
        # X shape : (n_samples, 6000) → (n_samples, 1, 6000) pour CNN 1D
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Split train/val depuis le train set
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.15, random_state=42, stratify=y_train_raw
)

# Datasets
train_dataset = ECGDataset(X_tr,          y_tr)
val_dataset   = ECGDataset(X_val,         y_val)
test_dataset  = ECGDataset(X_test_raw,    y_test_raw)

# DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0)

print('✅ Datasets créés :')
print(f'   Train   : {len(train_dataset):,} segments')
print(f'   Val     : {len(val_dataset):,} segments')
print(f'   Test    : {len(test_dataset):,} segments')
print(f'   Shape X : {train_dataset[0][0].shape}  (channels, samples)')

## 3. Architecture CNN 1D

Architecture inspirée des papiers de détection d'apnée sur ECG :
- **3 blocs Conv1D** avec BatchNorm + ReLU + MaxPool
- **Global Average Pooling** pour réduire la dimension
- **2 couches FC** avec Dropout pour la classification

Cette architecture apprend directement les patterns temporels ECG  
(cycles cardiaques, irrégularités, variations RR) sans feature engineering manuel.

In [ ]:
class CNN1D_Apnea(nn.Module):
    """
    CNN 1D pour la détection d'apnée du sommeil.
    
    Input  : (batch, 1, 6000) — signal ECG 1 min à 100 Hz
    Output : (batch, 2)       — logits [Normal, Apnée]
    """
    def __init__(self, dropout=0.3):
        super(CNN1D_Apnea, self).__init__()
        
        # Bloc 1 : features basses fréquences (cycles cardiaques)
        self.block1 = nn.Sequential(
            nn.Conv1d(1,  32, kernel_size=50, stride=2, padding=25),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Dropout(dropout)
        )
        
        # Bloc 2 : features moyennes fréquences (complexes QRS)
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=10, stride=1, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Dropout(dropout)
        )
        
        # Bloc 3 : features hautes fréquences (détails morphologiques)
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Dropout(dropout)
        )
        
        # Global Average Pooling → vecteur fixe quelle que soit la longueur
        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Classifieur
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


# Instancier et vérifier
model = CNN1D_Apnea(dropout=0.3).to(DEVICE)

# Test forward pass
test_input = torch.randn(4, 1, 6000).to(DEVICE)
test_output = model(test_input)
print('✅ Architecture CNN 1D :')
print(f'   Input  : {test_input.shape}')
print(f'   Output : {test_output.shape}')
print()

# Compter les paramètres
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'   Paramètres entraînables : {n_params:,}')
print()
print(model)

## 4. Entraînement + MLflow

In [ ]:
def compute_class_weights(y):
    """Calcule les poids de classes pour gérer le déséquilibre."""
    n_normal = sum(y == 0)
    n_apnea  = sum(y == 1)
    total    = len(y)
    w_normal = total / (2 * n_normal)
    w_apnea  = total / (2 * n_apnea)
    return torch.tensor([w_normal, w_apnea], dtype=torch.float32).to(DEVICE)


def train_epoch(model, loader, optimizer, criterion):
    """Une époque d'entraînement."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        preds      = outputs.argmax(dim=1)
        correct    += (preds == y_batch).sum().item()
        total      += len(y_batch)
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    """Évaluation sur un loader."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_labels = [], []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            outputs  = model(X_batch)
            loss     = criterion(outputs, y_batch)
            probs    = torch.softmax(outputs, dim=1)[:, 1]
            
            total_loss += loss.item() * len(y_batch)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == y_batch).sum().item()
            total      += len(y_batch)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    avg_loss = total_loss / total
    acc      = correct / total
    auc      = roc_auc_score(all_labels, all_probs)
    
    return avg_loss, acc, auc, np.array(all_probs), np.array(all_labels)


print('✅ Fonctions d\'entraînement définies')

In [ ]:
# Hyperparamètres
N_EPOCHS   = 30
LR         = 1e-3
DROPOUT    = 0.3

# Poids de classes
class_weights = compute_class_weights(y_tr)
print(f'Poids classes : Normal={class_weights[0]:.3f} | Apnée={class_weights[1]:.3f}')

print(f'\n🚀 Entraînement CNN 1D — {N_EPOCHS} époques')
print('=' * 50)

with mlflow.start_run(run_name='ECG_CNN1D_subject_split') as run:
    
    # Log params
    mlflow.log_params({
        'model'         : 'CNN1D',
        'n_epochs'      : N_EPOCHS,
        'learning_rate' : LR,
        'batch_size'    : BATCH_SIZE,
        'dropout'       : DROPOUT,
        'optimizer'     : 'Adam',
        'split_method'  : 'by_subject',
        'n_train_subj'  : len(TRAIN_SUBJECTS),
        'n_test_subj'   : len(TEST_SUBJECTS),
        'input_len'     : SEG_SAMPLES,
        'preprocessing' : 'bandpass + z-score intra-subject',
    })
    
    # Modèle, optimizer, loss
    model     = CNN1D_Apnea(dropout=DROPOUT).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=5, factor=0.5, verbose=True
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Historique
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   [], 'val_auc': []
    }
    
    best_val_auc  = 0
    best_model_state = None
    patience_counter = 0
    EARLY_STOP = 10
    
    for epoch in range(N_EPOCHS):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader,
                                             optimizer, criterion)
        # Val
        val_loss, val_acc, val_auc, _, _ = evaluate(model, val_loader, criterion)
        
        # Scheduler
        scheduler.step(val_auc)
        
        # Sauvegarder meilleur modèle
        if val_auc > best_val_auc:
            best_val_auc     = val_auc
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Log MLflow
        mlflow.log_metrics({
            'train_loss': train_loss,
            'train_acc' : train_acc,
            'val_loss'  : val_loss,
            'val_acc'   : val_acc,
            'val_auc'   : val_auc,
        }, step=epoch)
        
        # Historique
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:2d}/{N_EPOCHS} | '
                  f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} AUC: {val_auc:.4f}'
                  + (' ⭐' if val_auc == best_val_auc else ''))
        
        # Early stopping
        if patience_counter >= EARLY_STOP:
            print(f'  Early stopping à l\'époque {epoch+1}')
            break
    
    print(f'\n✅ Meilleure Val AUC : {best_val_auc:.4f}')
    
    # Charger meilleur modèle
    model.load_state_dict(best_model_state)
    
    # Évaluation finale sur test (split par sujet)
    test_loss, test_acc, test_auc, test_probs, test_labels = evaluate(
        model, test_loader, criterion
    )
    test_preds = (test_probs > 0.5).astype(int)
    test_f1    = f1_score(test_labels, test_preds, average='weighted')
    test_f1_ap = f1_score(test_labels, test_preds, pos_label=1)
    
    mlflow.log_metrics({
        'test_accuracy'  : test_acc,
        'test_auc_roc'   : test_auc,
        'test_f1_weighted': test_f1,
        'test_f1_apnea'  : test_f1_ap,
        'best_val_auc'   : best_val_auc,
    })
    
    # Sauvegarder modèle
    cnn_path = MODELS_DIR / 'somnia_cnn_ecg.pt'
    torch.save({
        'model_state_dict': best_model_state,
        'model_config'    : {'dropout': DROPOUT},
        'test_auc'        : test_auc,
        'train_subjects'  : TRAIN_SUBJECTS,
        'test_subjects'   : TEST_SUBJECTS,
    }, cnn_path)
    mlflow.log_artifact(str(cnn_path))
    
    run_id_cnn = run.info.run_id

print(f'\n📊 Résultats CNN — Split par sujet (sujets jamais vus) :')
print(f'   Test Accuracy : {test_acc:.4f}')
print(f'   Test AUC-ROC  : {test_auc:.4f}')
print(f'   Test F1 weighted : {test_f1:.4f}')
print(f'   Test F1 Apnée    : {test_f1_ap:.4f}')
print(f'\n   Modèle sauvegardé : {cnn_path}')

## 5. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Courbes d\'apprentissage — CNN 1D', fontsize=12, fontweight='bold')

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], color='#2E4057',
              label='Train', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   color='#E84855',
              label='Val', linewidth=2, linestyle='--')
axes[0].set_title('Loss')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], color='#2E4057',
              label='Train', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'],   color='#E84855',
              label='Val', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

# AUC
axes[2].plot(epochs_range, history['val_auc'], color='#048A81',
              label='Val AUC', linewidth=2)
axes[2].axhline(best_val_auc, color='#048A81', linestyle=':',
                 alpha=0.7, label=f'Best AUC={best_val_auc:.3f}')
axes[2].axhline(0.704, color='#EF8354', linestyle='--',
                 alpha=0.7, label='RF AUC=0.704')
axes[2].set_title('AUC-ROC (validation)')
axes[2].set_xlabel('Époque')
axes[2].set_ylabel('AUC-ROC')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
fig.savefig('../data/figures/cnn_learning_curves.png', dpi=150, bbox_inches='tight')

with mlflow.start_run(run_id=run_id_cnn):
    mlflow.log_artifact('../data/figures/cnn_learning_curves.png')

plt.show()
print('✅ Figure sauvegardée')

## 6. Évaluation détaillée + Comparaison RF vs CNN

In [ ]:
# Rapport de classification
print('=== Rapport CNN — Test set (sujets jamais vus) ===')
print(classification_report(test_labels, test_preds,
                             target_names=['Normal', 'Apnée']))

# Figure complète
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Évaluation CNN 1D — Détection apnée (split par sujet)',
              fontsize=12, fontweight='bold')

# Matrice de confusion
cm = confusion_matrix(test_labels, test_preds, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Reds',
            xticklabels=['Normal', 'Apnée'],
            yticklabels=['Normal', 'Apnée'], ax=axes[0])
axes[0].set_title('Matrice de confusion CNN')
axes[0].set_ylabel('Vrai label')
axes[0].set_xlabel('Label prédit')

# Courbe ROC
fpr_cnn, tpr_cnn, _ = roc_curve(test_labels, test_probs)
# Courbe ROC RF (valeurs du notebook 03)
rf_auc = 0.7035

axes[1].plot(fpr_cnn, tpr_cnn, color='#E84855', linewidth=2,
              label=f'CNN 1D     AUC={test_auc:.3f}')
# Approximation RF
fpr_rf = [0, 0.15, 0.30, 0.50, 1]
tpr_rf = [0, 0.55, 0.72, 0.85, 1]
axes[1].plot(fpr_rf, tpr_rf, color='#048A81', linewidth=2,
              linestyle='--', label=f'RF features AUC={rf_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', alpha=0.4, linewidth=1)
axes[1].fill_between(fpr_cnn, tpr_cnn, alpha=0.08, color='#E84855')
axes[1].set_xlabel('Taux faux positifs')
axes[1].set_ylabel('Taux vrais positifs')
axes[1].set_title('Courbe ROC — CNN vs RF')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

# Comparaison RF vs CNN — toutes métriques
metrics   = ['Accuracy', 'AUC-ROC', 'F1 weighted', 'F1 Apnée']
vals_rf   = [0.6885, 0.7035, 0.6853, 0.5838]
vals_cnn  = [test_acc, test_auc, test_f1, test_f1_ap]

x = np.arange(len(metrics))
w = 0.35
axes[2].bar(x - w/2, vals_rf,  w, label='RF + features', color='#048A81', alpha=0.85)
axes[2].bar(x + w/2, vals_cnn, w, label='CNN 1D',        color='#E84855', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics, rotation=15)
axes[2].set_ylim(0, 1.1)
axes[2].set_ylabel('Score')
axes[2].set_title('RF vs CNN — Split par sujet')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)
axes[2].axhline(0.5, color='gray', linestyle=':', alpha=0.5)

for i, (vr, vc) in enumerate(zip(vals_rf, vals_cnn)):
    axes[2].text(i - w/2, vr + 0.02, f'{vr:.3f}',
                  ha='center', fontsize=8, color='#048A81', fontweight='bold')
    axes[2].text(i + w/2, vc + 0.02, f'{vc:.3f}',
                  ha='center', fontsize=8, color='#E84855', fontweight='bold')

plt.tight_layout()
fig.savefig('../data/figures/cnn_vs_rf_comparison.png', dpi=150, bbox_inches='tight')

with mlflow.start_run(run_id=run_id_cnn):
    mlflow.log_artifact('../data/figures/cnn_vs_rf_comparison.png')

plt.show()
print('✅ Figure sauvegardée')

## Synthèse

In [ ]:
print('=' * 60)
print('SYNTHÈSE — CNN 1D vs Random Forest')
print('=' * 60)
print()
print('Validation : split par sujet (sujets jamais vus en train)')
print()
print(f'{"Métrique":<20} {"RF + features":>15} {"CNN 1D":>15} {"Delta":>10}')
print('-' * 62)

metrics_comp = [
    ('Accuracy',     0.6885, test_acc),
    ('AUC-ROC',      0.7035, test_auc),
    ('F1 weighted',  0.6853, test_f1),
    ('F1 Apnée',     0.5838, test_f1_ap),
]

for name, rf_val, cnn_val in metrics_comp:
    delta = cnn_val - rf_val
    arrow = '↑' if delta > 0 else '↓'
    print(f'{name:<20} {rf_val:>15.4f} {cnn_val:>15.4f} '
          f'{arrow}{abs(delta):>8.4f}')

print()
if test_auc > 0.7035:
    print('✅ Le CNN améliore la généralisation inter-sujets')
    print('   → Modèle CNN recommandé pour la production ECG')
else:
    print('⚠️  Le CNN ne surpasse pas le RF sur ce dataset')
    print('   → RF reste le modèle de production ECG')
    print('   → Plus d\'epochs ou d\'augmentation de données pourraient aider')

print()
print('📁 Modèles disponibles :')
for f in sorted(MODELS_DIR.glob('*')):
    print(f'   {f.name:<45} {f.stat().st_size/1024:.0f} Ko')

print()
print('🔜 PROCHAINE ÉTAPE : App FastAPI + déploiement HuggingFace')
print('   → Endpoints /predict/sleep-stage (EEG)')
print('   → Endpoints /predict/apnea (ECG — meilleur modèle)')
print('   → Dashboard Streamlit')
print('   → CI/CD GitHub Actions')